In [1]:
# Importing Libraries
import os
from typing import Dict, List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def parse_any_date(s: str) -> pd.Timestamp:
    """Parse common date formats; fallback to day-first."""
    s = str(s).strip()
    for fmt in ("%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            continue
    return pd.to_datetime(s, dayfirst=True, errors="raise")

def timestring_kind(s: str) -> str:
    """
    Classify time string:
    - 'hms' -> HH:MM:SS(.fff)
    - 'ms'  -> MM:SS
    - 'sec' -> seconds since midnight
    """
    s = str(s).strip()
    if s.count(":") == 2:
        return "hms"
    if s.count(":") == 1:
        left = s.split(":")[0]
        try:
            float(left); return "ms"
        except Exception:
            return "unknown"
    try:
        float(s); return "sec"
    except Exception:
        return "unknown"

def build_casas_ts_with_wrap(df_day: pd.DataFrame, date_str: str, time_col="Time") -> pd.Series:
    """
    Build timestamps for a single date. Supports:
    - HH:MM:SS(.fff), MM:SS (with wrap), or seconds since midnight.
    """
    base = parse_any_date(date_str).floor("D")
    tk = df_day[time_col].astype(str).map(timestring_kind)

    # HH:MM:SS(.fff)
    if (tk == "hms").any():
        ts = pd.to_datetime(base.strftime("%m/%d/%Y") + " " + df_day[time_col].astype(str),
                            format="%m/%d/%Y %H:%M:%S.%f", errors="coerce")
        ts = ts.fillna(pd.to_datetime(base.strftime("%m/%d/%Y") + " " + df_day[time_col].astype(str),
                                      format="%m/%d/%Y %H:%M:%S", errors="coerce"))
        if ts.notna().any():
            return ts

    # MM:SS with wrapping
    if (tk == "ms").any():
        mins = []
        for s in df_day[time_col].astype(str):
            m, sec = s.split(":")
            mins.append(float(m) + float(sec)/60.0)
        mins = np.array(mins, dtype=float)
        wraps = np.zeros_like(mins)
        w = 0
        for i in range(1, len(mins)):
            if mins[i] + 1e-6 < mins[i-1]:
                w += 1
            wraps[i] = w
        total_minutes = mins + 60.0 * wraps
        return base + pd.to_timedelta(total_minutes, unit="m")

    # Seconds since midnight
    if (tk == "sec").any():
        sec = pd.to_numeric(df_day[time_col], errors="coerce").fillna(0).to_numpy()
        return base + pd.to_timedelta((sec % 86400), unit="s")

    return pd.to_datetime(base.strftime("%m/%d/%Y") + " " + df_day[time_col].astype(str), errors="coerce")

# Activity name handling
def normalize_act_name(x: str) -> str:
    """Normalize labels to a compact, comparable set."""
    if x is None:
        return ""
    s = str(x).strip().lower().replace('"', '').replace("'", "").replace(" ", "_")
    base = s.split("=", 1)[0]  # drop ='begin' etc.
    mapping = {
        # meals
        "eat_breakfast":"breakfast","cook_breakfast":"breakfast","wash_breakfast_dishes":"breakfast","breakfast":"breakfast",
        "eat_lunch":"lunch","cook_lunch":"lunch","wash_lunch_dishes":"lunch","lunch":"lunch",
        "eat_dinner":"dinner","cook_dinner":"dinner","wash_dishes":"dinner","dinner":"dinner",
        # hygiene / toilet
        "toileting":"toileting","toilet":"toileting","personal_hygiene":"toileting","bed_toilet_transition":"toileting",
        # showering
        "bathe":"showering","shower":"showering","showering":"showering","groom":"showering",
        # leisure
        "watchtv":"watchingtv","watch_tv":"watchingtv","relax":"watchingtv","watchingtv":"watchingtv","read":"watchingtv","phone":"watchingtv",
        # sleep/nap
        "sleep":"sleeping","sleeping":"sleeping","napping":"napping","nap":"napping",
        # mobility / in-out
        "walk":"walking","walking":"walking","leave_home":"walking","enter_home":"walking",
        # snacks / meds
        "drink":"snacking","morning_meds":"snacking","evening_meds":"snacking",
    }
    return mapping.get(base, base)

def _pick(df: pd.DataFrame, want: str) -> str:
    """Pick column by name (case/space-insensitive) or substring."""
    want_key = want.strip().lower()
    lookup = {c.strip().lower(): c for c in df.columns}
    if want_key in lookup:
        return lookup[want_key]
    for c in df.columns:
        if want_key in c.strip().lower():
            return c
    raise KeyError(f"Column like '{want}' not found in: {list(df.columns)}")

# Loaders
def extract_casas_begin_activities(casas_csv: str, target_date: str,
                                   act_col_like: str = "Activity Label") -> pd.DataFrame:
    """CASAS: filter one date and keep only 'begin' markers."""
    df = pd.read_csv(casas_csv)

    date_col = _pick(df, "date")
    time_col = _pick(df, "time")
    act_col  = _pick(df, act_col_like)

    tgt = parse_any_date(target_date).date()
    df["_date"] = df[date_col].map(lambda x: parse_any_date(x).date())
    df = df[df["_date"] == tgt].copy()
    if df.empty:
        raise ValueError(f"No rows for {tgt} in {casas_csv}")

    mask_begin = df[act_col].astype(str).str.contains('="begin"', case=False, na=False)
    df = df[mask_begin].copy()

    df["ts"] = build_casas_ts_with_wrap(df, parse_any_date(target_date).strftime("%m/%d/%Y"), time_col=time_col)
    df["activity"] = df[act_col].map(normalize_act_name)
    df = df.dropna(subset=['ts']).copy()
    return df[["ts","activity"]].sort_values("ts").reset_index(drop=True)

def load_sharon_day(sharon_csv: str, target_date: str,
                    time_col: str = "time_of_the_day", name_col: str = "activity_name") -> pd.DataFrame:
    """SHARON: load one CSV and stamp onto target date."""
    sdf = pd.read_csv(sharon_csv)
    day = parse_any_date(target_date)

    t = sdf[time_col].astype(str).str.strip()
    t = np.where(t.str.count(":")==1, t + ":00", t)  # pad MM:SS -> MM:SS:00

    sdf["ts"] = pd.to_datetime(day.strftime("%Y-%m-%d") + " " + t, errors="coerce")
    sdf["activity"] = sdf[name_col].astype(str).map(normalize_act_name)
    sdf = sdf.dropna(subset=['ts']).copy()
    return sdf[["ts","activity"]].sort_values("ts").reset_index(drop=True)

def collapse_sessions(df: pd.DataFrame, gap_minutes: int = 60) -> pd.DataFrame:
    """Count sessions per activity using a simple gap rule."""
    df = df.sort_values("ts")
    rows = []
    for a, g in df.groupby("activity"):
        g = g.sort_values("ts").copy()
        diffs = g["ts"].diff().dt.total_seconds().div(60)
        starts = g.loc[(diffs.isna()) | (diffs > gap_minutes), "ts"]
        rows.append({"activity": a, "sessions": int(starts.size)})
    return pd.DataFrame(rows).sort_values("activity")

def compare_one_day(casas_csv: str, sharon_csv: str, target_date: str, gap_minutes: int = 60) -> Dict:
    """Run the single-day pipeline and return summary + counts table (no P/R/F1)."""
    casas_begin = extract_casas_begin_activities(casas_csv, target_date)
    sharon_day  = load_sharon_day(sharon_csv, target_date)

    cas = collapse_sessions(casas_begin, gap_minutes=gap_minutes).rename(columns={'sessions':'casas_sessions'})
    sha = collapse_sessions(sharon_day,  gap_minutes=gap_minutes).rename(columns={'sessions':'sharon_sessions'})
    table = (cas.merge(sha, on='activity', how='outer')
               .fillna(0)
               .astype({'casas_sessions': int, 'sharon_sessions': int}))

    cas_set = set(table.loc[table.casas_sessions > 0, 'activity'])
    sha_set = set(table.loc[table.sharon_sessions > 0, 'activity'])

    jaccard_presence = (len(cas_set & sha_set) / len(cas_set | sha_set)) if (cas_set | sha_set) else float('nan')
    table["min_cnt"] = table[["casas_sessions","sharon_sessions"]].min(axis=1)
    table["max_cnt"] = table[["casas_sessions","sharon_sessions"]].max(axis=1)
    weighted_jaccard = (table["min_cnt"].sum() / table["max_cnt"].sum()) if table["max_cnt"].sum() > 0 else np.nan

    return dict(
        date=target_date,
        sharon_file=os.path.basename(sharon_csv),
        jaccard_presence=jaccard_presence,
        weighted_jaccard=weighted_jaccard,
        session_table=table.sort_values('activity')[["activity","casas_sessions","sharon_sessions"]]
    )


# Run multiple days
if __name__ == "__main__":
    # Edit these paths for your environment
    CASAS_CSV  = '/content/drive/MyDrive/Dissertation/rw105.csv'
    SHARON_DIR = '/content/drive/MyDrive/Dissertation/30 days_activity output'

    # List any (date, SHARON-day-file) pairs you want to compare
    runs: List[tuple] = [
        ("27/10/2014", os.path.join(SHARON_DIR, "DAY2.csv")),
        ("20/10/2014", os.path.join(SHARON_DIR, "DAY4.csv")),
        # add more here...
    ]

    # Quick path checks
    if not os.path.exists(CASAS_CSV):
        raise FileNotFoundError(f"CASAS file not found: {CASAS_CSV}")
    for _, sharon_file in runs:
        if not os.path.exists(sharon_file):
            files = [f for f in os.listdir(SHARON_DIR) if f.lower().endswith('.csv')]
            raise FileNotFoundError(f"SHARON file not found: {sharon_file}\nAvailable: {files}")

    summaries = []
    for date_str, sharon_file in runs:
        print(f"\n=== {date_str} vs {os.path.basename(sharon_file)} ===")
        res = compare_one_day(CASAS_CSV, sharon_file, date_str, gap_minutes=60)

        # Print quick metrics (no P/R/F1)
        print(f"Presence Jaccard: {res['jaccard_presence']:.3f} | "
              f"Weighted Jaccard: {res['weighted_jaccard']:.3f}")

        # Per-activity session counts
        print(res["session_table"].to_string(index=False))

        # Keep for a summary table
        summaries.append({k:v for k,v in res.items() if k != "session_table"})

    # Side-by-side summary across runs
    summary_df = pd.DataFrame(summaries).sort_values("date")
    print("\n=== Summary across days ===")
    print(summary_df.to_string(index=False))



=== 27/10/2014 vs DAY2.csv ===
Presence Jaccard: 0.667 | Weighted Jaccard: 0.414
     activity  casas_sessions  sharon_sessions
    breakfast               1                1
       dinner               2                1
        dress               2                0
        lunch               1                1
      napping               0                1
    showering               2                1
     sleeping               2                2
     snacking               2                0
    toileting               5                3
      walking               4                1
   watchingtv               5                2
work_at_table               2                0

=== 20/10/2014 vs DAY4.csv ===
Presence Jaccard: 0.500 | Weighted Jaccard: 0.273
     activity  casas_sessions  sharon_sessions
    breakfast               1                1
     cleaning               0                1
         cook               1                0
       dinner               1        